# U-Net Practical Image Segmentation Model Training

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

# Simple config
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 10
LR = 0.001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class SimpleDataset(Dataset):
    def __init__(self, images_dir, masks_dir):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((IMG_SIZE, IMG_SIZE)),  # forces uniform size
            transforms.ToTensor()
        ])

        imgs = list(self.images_dir.glob('*.jpg'))
        self.pairs = [img for img in imgs
                     if (self.masks_dir / f"{img.stem}_mask.png").exists()]
        print(f" Dataset ready: {len(self.pairs)} pairs")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path = self.pairs[idx]
        mask_path = self.masks_dir / f"{img_path.stem}_mask.png"

        # Load image (BGR→RGB)
        img = cv2.imread(str(img_path))[...,::-1]

        # Load mask and normalize
        mask = cv2.imread(str(mask_path), 0)
        mask = (mask > 127).astype(np.float32)  # Binary 0/1

        # Transform RESIZES everything to 128x128
        img = self.transform(img)
        mask = self.transform(mask)[0][None]  # 1x128x128

        return img, mask

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1):
        super().__init__()
        # Encoder (Contracting Path)
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = nn.MaxPool2d(2)
        self.conv1 = DoubleConv(64, 128)
        self.down2 = nn.MaxPool2d(2)
        self.conv2 = DoubleConv(128, 256)
        self.down3 = nn.MaxPool2d(2)
        self.conv3 = DoubleConv(256, 512)
        self.down4 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder (Expansive Path)
        self.up1 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.conv4 = DoubleConv(1024, 512)  # 512+512 skip
        self.up2 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv5 = DoubleConv(512, 256)   # 256+256 skip
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv6 = DoubleConv(256, 128)   # 128+128 skip
        self.up4 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv7 = DoubleConv(128, 64)    # 64+64 skip

        self.outc = nn.Conv2d(64, n_classes, 1)

    def forward(self, x):
        # Encoder
        x1 = self.inc(x)                    # 64
        x2 = self.conv1(self.down1(x1))     # 128
        x3 = self.conv2(self.down2(x2))     # 256
        x4 = self.conv3(self.down3(x3))     # 512
        x5 = self.bottleneck(self.down4(x4)) # 1024

        # Decoder with skip connections
        x = self.up1(x5)                    # 512
        x = torch.cat([x, x4], dim=1)       # 1024
        x = self.conv4(x)                   # 512

        x = self.up2(x)                     # 256
        x = torch.cat([x, x3], dim=1)       # 512
        x = self.conv5(x)                   # 256

        x = self.up3(x)                     # 128
        x = torch.cat([x, x2], dim=1)       # 256
        x = self.conv6(x)                   # 128

        x = self.up4(x)                     # 64
        x = torch.cat([x, x1], dim=1)       # 128
        x = self.conv7(x)                   # 64

        return torch.sigmoid(self.outc(x))  # Binary segmentation output


In [8]:
# Setup
dataset = SimpleDataset(
    '/content/drive/MyDrive/COCO2017_SAMPLE/train2017',
    '/content/drive/MyDrive/COCO2017_SAMPLE/mask_train2017'
)

train_loader = DataLoader(dataset, BATCH_SIZE, shuffle=True, num_workers=2)

 Dataset ready: 1460 pairs


In [9]:
model = UNet(n_channels=3, n_classes=1).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCELoss()

print(f"🚀 Training {len(dataset)} images")

# Training loop
for epoch in range(EPOCHS):
    model.train()
    loss_total = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for imgs, masks in pbar:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, masks)
        loss.backward()
        optimizer.step()

        loss_total += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    print(f"Epoch {epoch+1} Avg Loss: {loss_total/len(train_loader):.4f}")

torch.save(model.state_dict(), 'unet_seg.pth')
print("✅ Saved model!")



🚀 Training 1460 images


Epoch 1: 100%|██████████| 46/46 [10:39<00:00, 13.91s/it, loss=0.4045]


Epoch 1 Avg Loss: 0.5532


Epoch 2: 100%|██████████| 46/46 [00:35<00:00,  1.31it/s, loss=0.4606]


Epoch 2 Avg Loss: 0.5022


...

Epoch 8 Avg Loss: 0.4452


Epoch 9: 100%|██████████| 46/46 [00:33<00:00,  1.37it/s, loss=0.5738]


Epoch 9 Avg Loss: 0.4383


Epoch 10: 100%|██████████| 46/46 [00:35<00:00,  1.31it/s, loss=0.4092]


Epoch 10 Avg Loss: 0.4300
✅ Saved model!


In [19]:
# Test function
def remove_bg(img_path, model_path='unet_seg.pth'):
    model = UNet()
    model.load_state_dict(torch.load(model_path))
    model.to(DEVICE).eval()

    img = cv2.imread(img_path)
    orig = img.copy()
    h, w = img.shape[:2]

    # Preprocess
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor()
    ])
    img_tensor = transform(img[...,::-1]).unsqueeze(0).to(DEVICE)

    # Predict
    with torch.no_grad():
        mask = model(img_tensor)[0,0].cpu().numpy()
        mask = cv2.resize(mask, (w, h)) > 0.5

    # Apply
    result = orig * mask[:,:,None]
    cv2.imwrite('result_no_bg.jpg', result)
    print("🎯 Saved: result_no_bg.jpg")

# Test
test_img = "/content/drive/MyDrive/COCO2017_SAMPLE/train2017/000000000036.jpg"
remove_bg(test_img)

🎯 Saved: result_no_bg.jpg
